# LAB 01 — Experiments

Notebook cho ba phần đầu của LAB 01: setup dataset, kiểm tra sparse representation và vocabulary inspection.

## 1. Setup and Dataset

Dataset không được commit trong repository. Hãy đặt corpus được cung cấp vào `lab01/data/` hoặc cấu hình `DATA_PATH` trong cell bên dưới. Notebook không tự tìm file bên ngoài project và không giả định format khi dataset chưa được cấu hình.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

# Set this to the provided corpus path after placing it in the project.
# Example: DATA_PATH = Path("data") / "provided-corpus-file"
DATA_PATH = None

DATA_AVAILABLE = False
documents = []
records = []
if DATA_PATH is None:
    print("Dataset not configured. Set DATA_PATH to the provided 30K corpus.")
else:
    DATA_PATH = Path(DATA_PATH)
    if not DATA_PATH.is_file():
        raise FileNotFoundError(f"Configured dataset path does not exist: {DATA_PATH}")
    raise NotImplementedError(
        "Configure the loader after confirming the provided corpus format."
    )


## 2. Experiment 1 — Sparse Representation

Pipeline: raw documents → tokenizer → CountVectorizer → normalized TF → IDF → TF-IDF matrix. The large matrix remains sparse throughout. `CountVectorizer()` uses sklearn's default tokenization and lowercasing behavior.

TODO for the preprocessing ablation: explicitly control lowercasing and tokenization so that Pipeline A/B/C do not apply preprocessing twice or hide it inside the vectorizer.

In [ ]:
if DATA_AVAILABLE:
    try:
        from sklearn.feature_extraction.text import CountVectorizer
        from sklearn.preprocessing import normalize
        from scipy import sparse
        SKLEARN_AVAILABLE = True
    except ImportError as error:
        SKLEARN_AVAILABLE = False
        print(f"scikit-learn is required for Experiment 1: {error}")
else:
    SKLEARN_AVAILABLE = False

if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    vectorizer = CountVectorizer()
    count_matrix = vectorizer.fit_transform(documents)
    tf_matrix = normalize(count_matrix, norm="l1", axis=1, copy=True)

    document_frequency = np.asarray(count_matrix.getnnz(axis=0)).ravel()
    number_of_documents = count_matrix.shape[0]
    idf_values = np.log(number_of_documents / document_frequency)
    tfidf_matrix = tf_matrix.multiply(idf_values).tocsr()
    tfidf_matrix.eliminate_zeros()
    feature_names = vectorizer.get_feature_names_out()

    number_of_documents, vocabulary_size = tfidf_matrix.shape
    nnz = tfidf_matrix.nnz
    total_entries = number_of_documents * vocabulary_size
    sparsity = 1 - nnz / total_entries if total_entries else 0.0

    summary = pd.DataFrame({
        "metric": [
            "Number of documents",
            "Vocabulary size",
            "TF-IDF matrix shape",
            "Non-zero entries (nnz)",
            "Sparsity",
        ],
        "value": [
            number_of_documents,
            vocabulary_size,
            tfidf_matrix.shape,
            nnz,
            sparsity,
        ],
    })
    display(summary)
    print(f"Count matrix sparse: {sparse.issparse(count_matrix)}")
    print(f"TF matrix sparse: {sparse.issparse(tf_matrix)}")
    print(f"TF-IDF matrix sparse: {sparse.issparse(tfidf_matrix)}")
    print("CountVectorizer uses sklearn's default tokenization and lowercasing.")
elif not DATA_AVAILABLE:
    print("Experiment 1 skipped because the dataset is not available.")
else:
    print("Experiment 1 skipped because scikit-learn is not available.")


## 3. Vocabulary Inspection

The tables below use the feature ordering returned by the fitted vectorizer. Document frequency means the number of documents containing a term, not the total number of occurrences.

In [ ]:
def get_top_df_terms(document_frequency, feature_names, top_k=20):
    """Return terms ranked by document frequency."""
    table = pd.DataFrame({
        "term": feature_names,
        "document_frequency": document_frequency,
    })
    return table.sort_values("document_frequency", ascending=False).head(top_k).reset_index(drop=True)

def get_top_idf_terms(feature_names, idf, top_k=20):
    """Return terms ranked by the fitted transformer IDF values."""
    table = pd.DataFrame({"term": feature_names, "idf": idf})
    return table.sort_values("idf", ascending=False).head(top_k).reset_index(drop=True)

def get_top_tfidf_terms(tfidf_matrix, feature_names, document_index, top_k=20):
    """Return non-zero TF-IDF terms for one selected document."""
    row = tfidf_matrix.getrow(document_index)
    table = pd.DataFrame({
        "term": feature_names[row.indices],
        "tfidf": row.data,
    })
    return table.sort_values("tfidf", ascending=False).head(top_k).reset_index(drop=True)

if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    top_df_terms = get_top_df_terms(document_frequency, feature_names)
    top_idf_terms = get_top_idf_terms(feature_names, idf_values)

    SELECTED_DOC_INDEX = 0
    top_tfidf_terms = get_top_tfidf_terms(
        tfidf_matrix, feature_names, SELECTED_DOC_INDEX
    )

    print("Top 20 terms by document frequency")
    display(top_df_terms)
    print("Top 20 terms by IDF")
    display(top_idf_terms)
    print(f"Top TF-IDF terms in document {SELECTED_DOC_INDEX}")
    print(documents[SELECTED_DOC_INDEX][:500])
    display(top_tfidf_terms)
elif not DATA_AVAILABLE:
    print("Vocabulary inspection skipped because the dataset is not available.")
else:
    print("Vocabulary inspection skipped because scikit-learn is not available.")


### Student analysis

TODO:
- Compare the three term lists.
- Does a frequent corpus term necessarily have high TF-IDF?
- Does a high-IDF term necessarily have high TF-IDF in every document?

## 4. Experiment 2 — Preprocessing Ablation

The three pipelines are intentionally simple. Pipeline C uses deterministic character trigrams as a lightweight subword tokenizer; no large language model is required.

In [ ]:
import re
from collections.abc import Callable

# A small, explicit stopword list keeps Pipeline B reproducible without
# adding a hidden external preprocessing dependency.
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by",
    "for", "from", "in", "is", "it", "of", "on", "or",
    "that", "the", "this", "to", "was", "were", "with",
}

def tokenize_with_punctuation(text: str) -> list[str]:
    return re.findall(r"\b\w+\b|[^\w\s]", str(text).lower(), flags=re.UNICODE)

def tokenize_words(text: str) -> list[str]:
    return re.findall(r"\b\w+\b", str(text).lower(), flags=re.UNICODE)

def preprocess_a(text: str) -> list[str]:
    """Lowercase and tokenize while retaining punctuation tokens."""
    return tokenize_with_punctuation(text)

def preprocess_b(text: str) -> list[str]:
    """Lowercase, remove punctuation, tokenize, and remove stopwords."""
    return [token for token in tokenize_words(text) if token not in STOPWORDS]

def preprocess_c(text: str) -> list[str]:
    """Normalize words into deterministic character-trigram subwords."""
    subwords: list[str] = []
    for word in tokenize_words(text):
        if len(word) <= 3:
            subwords.append(word)
            continue
        bounded_word = f"<{word}>"
        subwords.extend(
            bounded_word[index:index + 3]
            for index in range(len(bounded_word) - 2)
        )
    return subwords

def identity_analyzer(tokens: list[str]) -> list[str]:
    """Pass already-tokenized input to CountVectorizer unchanged."""
    return tokens

def make_token_vectorizer():
    if not SKLEARN_AVAILABLE:
        raise RuntimeError("scikit-learn is required for vectorization.")
    return CountVectorizer(
        analyzer=identity_analyzer,
        lowercase=False,
        preprocessor=None,
        tokenizer=None,
        token_pattern=None,
    )

def fit_tfidf_model(
    tokenized_documents: list[list[str]],
    raw_documents: list[str],
    preprocess: Callable[[str], list[str]],
    document_ids=None,
) -> dict:
    """Fit a sparse TF-IDF index using the same vocabulary for queries."""
    if not tokenized_documents:
        raise ValueError("Cannot fit an index on an empty corpus.")
    if not any(tokenized_documents):
        raise ValueError("All documents are empty after preprocessing.")

    vectorizer = make_token_vectorizer()
    count_matrix = vectorizer.fit_transform(tokenized_documents)
    tf_matrix = normalize(count_matrix, norm="l1", axis=1, copy=True)
    document_frequency = np.asarray(count_matrix.getnnz(axis=0)).ravel()
    idf_values = np.log(count_matrix.shape[0] / document_frequency)
    raw_tfidf_matrix = tf_matrix.multiply(idf_values).tocsr()
    raw_tfidf_matrix.eliminate_zeros()
    search_matrix = normalize(raw_tfidf_matrix, norm="l2", axis=1, copy=True)

    ids = list(range(len(raw_documents))) if document_ids is None else list(document_ids)
    if len(ids) != len(raw_documents):
        raise ValueError("document_ids must match the number of documents.")
    return {
        "vectorizer": vectorizer,
        "count_matrix": count_matrix,
        "tf_matrix": tf_matrix,
        "raw_tfidf_matrix": raw_tfidf_matrix,
        "tfidf_matrix": search_matrix,
        "idf_values": idf_values,
        "feature_names": vectorizer.get_feature_names_out(),
        "tokenized_documents": tokenized_documents,
        "raw_documents": list(raw_documents),
        "document_ids": ids,
        "id_to_index": {document_id: index for index, document_id in enumerate(ids)},
        "preprocess": preprocess,
    }


## 5. Document Search

Queries use the fitted vectorizer, vocabulary, and IDF values from the selected pipeline. Only the document-score vector is converted to dense form; the corpus matrix remains sparse.

In [ ]:
SEARCH_COLUMNS = ["rank", "document_id", "similarity", "document_preview"]

def empty_search_results() -> pd.DataFrame:
    return pd.DataFrame(columns=SEARCH_COLUMNS)

def search(query: str, model: dict, top_k: int = 5) -> pd.DataFrame:
    """Return ranked documents for a query using cosine similarity."""
    if top_k <= 0 or not str(query).strip():
        return empty_search_results()

    query_tokens = model["preprocess"](query)
    if not query_tokens:
        return empty_search_results()
    query_counts = model["vectorizer"].transform([query_tokens])
    query_tf = normalize(query_counts, norm="l1", axis=1, copy=True)
    query_tfidf = query_tf.multiply(model["idf_values"]).tocsr()
    query_tfidf.eliminate_zeros()
    if query_tfidf.nnz == 0:
        return empty_search_results()

    query_vector = normalize(query_tfidf, norm="l2", axis=1, copy=True)
    scores = model["tfidf_matrix"].dot(query_vector.T).toarray().ravel()
    ranking = np.argsort(-scores, kind="stable")[: min(top_k, len(scores))]
    rows = []
    for rank, index in enumerate(ranking, start=1):
        rows.append({
            "rank": rank,
            "document_id": model["document_ids"][index],
            "similarity": float(scores[index]),
            "document_preview": model["raw_documents"][index][:300],
        })
    return pd.DataFrame(rows, columns=SEARCH_COLUMNS)

EXAMPLE_QUERIES = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing",
]


## 6. Pipeline Comparison

Search performance remains unavailable until the student supplies relevance labels. The code reports `NaN` rather than fabricating a score.

In [ ]:
def compute_oov_rate(model: dict, queries: list[str]) -> float:
    vocabulary = set(model["feature_names"])
    total_tokens = 0
    oov_tokens = 0
    for query in queries:
        tokens = model["preprocess"](query)
        total_tokens += len(tokens)
        oov_tokens += sum(token not in vocabulary for token in tokens)
    return oov_tokens / total_tokens if total_tokens else float("nan")

def evaluate_pipeline(
    raw_documents: list[str],
    preprocess: Callable[[str], list[str]],
    queries: list[str],
    evaluation_set: dict,
) -> tuple[dict, dict]:
    tokenized_documents = [preprocess(document) for document in raw_documents]
    model = fit_tfidf_model(tokenized_documents, raw_documents, preprocess)
    matrix = model["raw_tfidf_matrix"]
    rows, columns = matrix.shape
    total_entries = rows * columns
    metrics = {
        "Vocabulary size": columns,
        "Average tokens/document": float(np.mean([len(tokens) for tokens in tokenized_documents])),
        "Matrix sparsity": 1 - matrix.nnz / total_entries if total_entries else 0.0,
        "OOV rate": compute_oov_rate(model, queries),
        "Search performance (MRR)": float("nan"),
    }
    if evaluation_set:
        retrieved = {
            query: search(query, model, top_k=5)["document_id"].tolist()
            for query in evaluation_set
        }
        metrics["Search performance (MRR)"] = mean_reciprocal_rank(retrieved, evaluation_set)
    return model, metrics

# Student supplies relevance labels before enabling metric calculation.
evaluation_set = {}

if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    pipeline_specs = {
        "Pipeline A": preprocess_a,
        "Pipeline B": preprocess_b,
        "Pipeline C": preprocess_c,
    }
    pipeline_models = {}
    pipeline_metrics = {}
    for pipeline_name, preprocess in pipeline_specs.items():
        pipeline_models[pipeline_name], pipeline_metrics[pipeline_name] = evaluate_pipeline(
            documents, preprocess, EXAMPLE_QUERIES, evaluation_set
        )
    comparison = pd.DataFrame(pipeline_metrics).reindex(
        [
            "Vocabulary size",
            "Average tokens/document",
            "Matrix sparsity",
            "OOV rate",
            "Search performance (MRR)",
        ]
    )
    display(comparison)
else:
    print("Pipeline comparison skipped because the dataset or scikit-learn is unavailable.")


### Student analysis

TODO: compare the metrics. Do not assume that more preprocessing is always better.

## 7. Evaluation Metrics

Enter relevance labels in `evaluation_set` before running this section. No document IDs are invented here.

In [ ]:
def precision_at_k(retrieved_ids, relevant_ids, k: int = 5) -> float:
    if k <= 0:
        return 0.0
    relevant = set(relevant_ids)
    retrieved = list(retrieved_ids)[:k]
    return sum(document_id in relevant for document_id in retrieved) / k

def recall_at_k(retrieved_ids, relevant_ids, k: int = 5) -> float:
    relevant = set(relevant_ids)
    if not relevant:
        return 0.0
    retrieved = set(list(retrieved_ids)[:k])
    return len(retrieved & relevant) / len(relevant)

def reciprocal_rank(retrieved_ids, relevant_ids) -> float:
    relevant = set(relevant_ids)
    for rank, document_id in enumerate(retrieved_ids, start=1):
        if document_id in relevant:
            return 1.0 / rank
    return 0.0

def mean_reciprocal_rank(retrieved_by_query: dict, evaluation_set: dict) -> float:
    if not evaluation_set:
        return float("nan")
    scores = [
        reciprocal_rank(retrieved_by_query.get(query, []), relevant_ids)
        for query, relevant_ids in evaluation_set.items()
    ]
    return float(np.mean(scores))

if not evaluation_set:
    print("Evaluation skipped: add student-provided relevance labels to evaluation_set.")


## 8. Results Export

Results are exported only when relevance labels are available.

In [ ]:
RESULT_COLUMNS = ["query", "rank", "document_id", "similarity", "relevant"]

def build_results_table(query_results: dict, evaluation_set: dict) -> pd.DataFrame:
    rows = []
    for query, result_table in query_results.items():
        relevant_ids = set(evaluation_set.get(query, []))
        for result in result_table.itertuples(index=False):
            rows.append({
                "query": query,
                "rank": result.rank,
                "document_id": result.document_id,
                "similarity": result.similarity,
                "relevant": result.document_id in relevant_ids,
            })
    return pd.DataFrame(rows, columns=RESULT_COLUMNS)

def results_path() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "lab01").is_dir():
            return candidate / "lab01" / "results.csv"
    return Path("results.csv")

if DATA_AVAILABLE and SKLEARN_AVAILABLE and evaluation_set:
    selected_model = pipeline_models["Pipeline A"]
    query_results = {
        query: search(query, selected_model, top_k=5)
        for query in evaluation_set
    }
    results_df = build_results_table(query_results, evaluation_set)
    results_df.to_csv(results_path(), index=False)
    display(results_df)
else:
    print("Results export skipped: relevance labels and a fitted model are required.")


## 9. Error Analysis Support

The helper below exposes retrieval evidence. The student must select and explain good results, poor results, and a failure case.

In [ ]:
def inspect_query(query: str, model: dict, expected_relevant=None, top_k: int = 5) -> dict:
    result_table = search(query, model, top_k=top_k)
    query_terms = set(model["preprocess"](query))
    overlaps = []
    for result in result_table.itertuples(index=False):
        index = model["id_to_index"][result.document_id]
        document_terms = set(model["tokenized_documents"][index])
        overlaps.append(len(query_terms & document_terms))
    if not result_table.empty:
        result_table = result_table.copy()
        result_table["lexical_overlap"] = overlaps
    return {
        "query": query,
        "expected_relevant": None if expected_relevant is None else list(expected_relevant),
        "retrieved": result_table,
    }


### Student error analysis

TODO: choose two good queries, two poor queries, and one important failure case.